# Daten einlesen

In [ ]:
import pandas as pd

path_to_dataset = '02 Myocardial infarction complications Database.csv'

df = pd.read_csv(path_to_dataset) # evtl mit , sep=';'
df

## Modify dataset
### List all column names

In [ ]:
print(df.columns.values)

### List column names and datatypes

In [ ]:
overview = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str),
    "nan_count": df.isna().sum(),
    "nan_percent": (df.isna().mean() * 100).round(2)
})

print(overview.values)

### Drop columns

In [ ]:
y = df['ZSN']
X = df.drop('ZSN', axis=1)

### Change datatype to categorical
Change the datatype of the column to categorical if it has less than 10 unique values

In [1]:
for col in df.columns:
    if df[col].nunique() < 10:
        try:
            df[col] = df[col].astype('float').astype('Int64').astype('category')
        except (ValueError, TypeError):
            df[col] = df[col].astype('category')
        print(f"Column '{col}' converted to 'category' dtype.")

NameError: name 'df' is not defined

### Automatic check for reasonable data ranges
Upload column names and feature description to your LLM and ask for a list / csv of acceptable value ranges for each feature.

Prompt:
Make a csv in with this structure
column,min,max,allowed_values

Fill them with acceptable value ranges for the given features. For values 0 and 1 write them as 0.0 and 1.0

For all those columns: {list of column names copy+paste from above}

In [ ]:
import pandas as pd

def validate_dataframe(df: pd.DataFrame, ranges_csv: str = "feature_ranges.csv") -> pd.DataFrame:
    """
    Compare a DataFrame against acceptable value ranges defined in a CSV.

    Expected CSV format (produced by the LLM):
        column,min,max,allowed_values
        age,0,120,
        gender,,,M|F
        score,0.0,1.0,
        blood_type,,,A+|A-|B+|B-|O+|O-|AB+|AB-

    Rules:
    - Numeric columns: provide min and/or max (leave allowed_values empty).
    - Categorical columns: provide pipe-separated allowed_values (leave min/max empty).

    Args:
        df:         DataFrame to validate.
        ranges_csv: Path to the CSV file with acceptable ranges.

    Returns:
        Summary DataFrame with one row per column:
            column           – column name
            total_non_null   – non-null values checked
            out_of_range     – count of values outside the acceptable range
            pct_out_of_range – percentage out of range
            rule_applied     – human-readable rule that was used
    """
    ranges = pd.read_csv(ranges_csv)

    records = []

    for _, rule in ranges.iterrows():
        col = rule["column"]

        if col not in df.columns:
            print(f"⚠️  '{col}' in CSV but not in DataFrame — skipped.")
            continue

        series = df[col].dropna()
        total = len(series)

        if total == 0:
            records.append(dict(column=col, total_non_null=0, out_of_range=0,
                                pct_out_of_range=0.0, rule_applied="no non-null values"))
            continue

        allowed_raw = rule.get("allowed_values")
        col_min     = rule.get("min")
        col_max     = rule.get("max")

        has_allowed = pd.notna(allowed_raw) and str(allowed_raw).strip() != ""
        has_min     = pd.notna(col_min)
        has_max     = pd.notna(col_max)

        if has_allowed:
            allowed_list = [v.strip() for v in str(allowed_raw).split(",")]
            mask_bad = ~series.astype(str).isin(allowed_list)
            rule_str = f"allowed_values={allowed_list}"
        else:
            mask_bad = pd.Series(False, index=series.index)
            parts = []
            if has_min:
                mask_bad |= pd.to_numeric(series, errors="coerce") < float(col_min)
                parts.append(f"min={col_min}")
            if has_max:
                mask_bad |= pd.to_numeric(series, errors="coerce") > float(col_max)
                parts.append(f"max={col_max}")
            rule_str = ", ".join(parts) if parts else "no bounds defined"

        n_bad = int(mask_bad.sum())
        records.append(dict(
            column=col,
            total_non_null=total,
            out_of_range=n_bad,
            pct_out_of_range=round(100 * n_bad / total, 2),
            rule_applied=rule_str,
        ))

    return (pd.DataFrame(records)
              .sort_values("out_of_range", ascending=False)
              .reset_index(drop=True))

summary = validate_dataframe(df, "feature_ranges.csv")
print(summary.to_string(index=False))

### Onehot encoding

In [ ]:
df_encoded = pd.get_dummies(df, drop_first=False)
# or
# df_encoded = pd.get_dummies(df, columns=['col1', 'col2'], drop_first=False)

### Train-test split

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(X_train.shape)

## EDA
### Overview

In [ ]:
X_train.describe()

### List missing values by column

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
columns = df.columns
print("Missing values in each column:")
for col, missing in zip(columns, missing_values):
    print(f"{col}: {missing}")

## Standardization

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = scaler.transform(X_test)
X_test = pd.DataFrame(X_test_scaled, columns=X_train.columns)

## Nan-value imputation
### Simple imputation

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='constant', fill_value=0)
# Possibilities: 'mean', 'median', 'most_frequent', 'constant'
X_train_imputed = imputer.fit_transform(X_train)
X_train = pd.DataFrame(X_train_imputed, columns=X_train.columns)
X_test = imputer.transform(X_test)
X_test = pd.DataFrame(X_test, columns=X_train.columns)

### KNN imputation

In [ ]:
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=5)
X_train_imputed = imputer.fit_transform(X_train)
X_train = pd.DataFrame(X_train_imputed, columns=X_train.columns)
X_test = imputer.transform(X_test)
X_test = pd.DataFrame(X_test, columns=X_train.columns)

## Models
### Linear regression

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)

# Explanation of coefficients
coefficients = model.coef_
feature_names = X_train.columns

# Sort by absolute coefficient value (descending)
features_and_coefficients = sorted(
    zip(feature_names, coefficients),
    key=lambda x: abs(x[1]),
    reverse=True
)

print("Top 10 features by absolute coefficient value:")
for feature, coefficient in features_and_coefficients[:10]:
    print(f"{feature}: {coefficient:.4f}")


### Logistic regression

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

# Explanation of coefficients
coefficients = model.coef_[0]  # coef_ is 2D for logistic regression, shape (1, n_features)
feature_names = X_train.columns

# Sort by absolute coefficient value (descending)
features_and_coefficients = sorted(
    zip(feature_names, coefficients),
    key=lambda x: abs(x[1]),
    reverse=True
)

print("Top 10 features by absolute coefficient value:")
for feature, coefficient in features_and_coefficients[:10]:
    print(f"{feature}: {coefficient:.4f}")

### Decision tree

In [ ]:
### Decision Tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

# Evaluation
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Feature importances
feature_names = X_train.columns
features_and_importances = sorted(
    zip(feature_names, model.feature_importances_),
    key=lambda x: x[1],
    reverse=True
)

print("Top 10 features by importance:")
for feature, importance in features_and_importances[:10]:
    print(f"{feature}: {importance:.4f}")

### Rule Based Model

In [ ]:
from imodels import RuleFitClassifier
from sklearn.ensemble import GradientBoostingClassifier
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    warnings.simplefilter("ignore", category=UserWarning)

model = RuleFitClassifier(
    tree_size=4,  # mean number of terminal nodes per tree (rule complexity)
    sample_fract=0.5,  # use 50% of samples per tree (reduces overfitting)
    max_rules=100,  # max number of rules + linear terms in final model
    memory_par=0.01,  # shrinkage factor between trees (lower = more conservative)
    lin_standardise=True,  # standardise linear terms as per Friedman Sec 3.2
    lin_trim_quantile=0.025,  # trim outliers before standardising linear terms
    exp_rand_tree_size=True,  # vary tree sizes around tree_size (more diverse rules)
    include_linear=True,  # include linear terms alongside rules
    cv=True,  # use cross-validation to select regularization strength
    random_state=42,
)

model.fit(X_train.values, y_train.values, feature_names=X_train.columns.tolist())

# Evaluation
y_pred = model.predict(X_test.values)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

#### Extract Rules
rules = model.get_rules()

rules = rules[rules['coef'] != 0].copy()
rules['abs_coef'] = rules['coef'].abs()
rules = rules.sort_values('abs_coef', ascending=False)

print(f"Total active rules: {len(rules)}")
print("\nTop 20 rules by importance:")
print(rules[['rule', 'coef', 'support', 'importance']].head(20).to_string(index=False))